In [ ]:
## Cell 1 · Imports

import pandas as pd
import numpy as np
import math
import osmnx as ox
from sklearn.neighbors import BallTree

df = pd.read_csv("csv/00_base_data.csv")
print(f"Loaded {len(df)} records")

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

In [ ]:
## Cell 2 · Fetch Subway Entrances from OSM

# Infer center and search radius from shop data + 1500 m buffer
center_lat = (df["lat"].max() + df["lat"].min()) / 2
center_lon = (df["lon"].max() + df["lon"].min()) / 2
max_dist = max(
    haversine(center_lat, center_lon, row["lat"], row["lon"])
    for _, row in df.iterrows()
)
search_radius = max_dist + 1500
print(f"Center: ({center_lat:.4f}, {center_lon:.4f})  Search radius: {search_radius:.0f} m")

print("Fetching subway entrances from OSM...")
subway = ox.features_from_point(
    (center_lat, center_lon),
    tags={"railway": "subway_entrance"},
    dist=search_radius
)

subway = subway[["geometry"]].copy()
subway["lat_s"] = subway.geometry.centroid.y
subway["lon_s"] = subway.geometry.centroid.x
subway = subway.dropna(subset=["lat_s", "lon_s"])
print(f"  {len(subway)} subway entrances found")

In [3]:
## Cell 3 · Match Each Shop to Nearest Subway Entrance

subway_coords = np.radians(subway[["lat_s", "lon_s"]].values)
tree = BallTree(subway_coords, metric="haversine")

shop_coords = np.radians(df[["lat", "lon"]].values)
distances, indices = tree.query(shop_coords, k=1)

# Convert from radians to meters
df["dist_subway_m"] = distances.flatten() * 6371000

print(f"✓ Done")
print(f"  Fill : {df['dist_subway_m'].notna().sum()}/{len(df)}")
print(f"  Mean : {df['dist_subway_m'].mean():.0f} m")
print(f"  Min  : {df['dist_subway_m'].min():.0f} m")
print(f"  Max  : {df['dist_subway_m'].max():.0f} m")

✓ Done
  Fill : 2915/2915
  Mean : 141 m
  Min  : 3 m
  Max  : 900 m


In [5]:
df["dist_subway_m"] = df["dist_subway_m"].round(2)

In [6]:
## Cell 4 · Save

df_out = df[["osm_id", "dist_subway_m"]]
df_out.to_csv("csv/10_subway_distance.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/10_subway_distance.csv")

Saved 2915 records to csv/10_subway_distance.csv
